# Offside Detection — Legacy v1 Classic ML Benchmark
## `02_train_model.ipynb`


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
BASE_DIR = Path('/content/drive/MyDrive/Offside_Detector')
data = np.load(BASE_DIR / 'features.npz', allow_pickle=True)
X, y = data['X'].astype(np.float32), data['y'].astype(np.int64)
feature_names = list(data['feature_names'])

df = pd.DataFrame(X, columns=feature_names)
df['y'] = y
display(pd.DataFrame({
    'Mean (Onside=0)': df[df['y'] == 0][feature_names].mean(),
    'Mean (Offside=1)': df[df['y'] == 1][feature_names].mean(),
    'Corr with y': [df[c].corr(df['y']) for c in feature_names],
}).round(4))


In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
folds = list(skf.split(X, y))
for i, (tr, va) in enumerate(folds, 1):
    print(f"Fold {i}: train={X[tr].shape}, val={X[va].shape}")


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

models = {
    'Logistic Regression': LogisticRegression(C=0.5, max_iter=1000, random_state=42),
    'SVM (RBF Kernel)': SVC(C=1.0, kernel='rbf', probability=True, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=4, min_samples_leaf=3, random_state=42),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=80, max_depth=2, learning_rate=0.05, random_state=42),
    'KNN (k=7)': KNeighborsClassifier(n_neighbors=7),
}

base_preds = (X[:, 0] > 0).astype(int)
ml_results = [{'Model': 'Baseline (diff_x > 0)', 'Accuracy': accuracy_score(y, base_preds) * 100,
               'F1-Score': f1_score(y, base_preds) * 100, 'ROC-AUC': roc_auc_score(y, X[:, 0]) * 100}]

for name, model in models.items():
    oof_preds, oof_probs = np.zeros(len(y), dtype=np.int64), np.zeros(len(y), dtype=np.float32)
    for tr, va in folds:
        scaler = StandardScaler()
        model.fit(scaler.fit_transform(X[tr]), y[tr])
        oof_preds[va] = model.predict(scaler.transform(X[va]))
        oof_probs[va] = model.predict_proba(scaler.transform(X[va]))[:, 1]
    ml_results.append({'Model': name, 'Accuracy': accuracy_score(y, oof_preds) * 100,
                       'F1-Score': f1_score(y, oof_preds) * 100, 'ROC-AUC': roc_auc_score(y, oof_probs) * 100})

display(pd.DataFrame(ml_results).sort_values('Accuracy', ascending=False).round(2))


In [ ]:
dbg = pd.read_csv(BASE_DIR / 'features_debug.csv')
dbg['wrong'] = (dbg['rule_pred'] != dbg['y'])
print(f"Baseline errors: {dbg['wrong'].sum()} / {len(dbg)}")
for name, model in models.items():
    tr_accs, va_accs = [], []
    for tr, va in folds:
        sc = StandardScaler()
        X_tr, X_va = sc.fit_transform(X[tr]), sc.transform(X[va])
        model.fit(X_tr, y[tr])
        tr_accs.append(accuracy_score(y[tr], model.predict(X_tr)) * 100)
        va_accs.append(accuracy_score(y[va], model.predict(X_va)) * 100)
    print(f"{name:22s} | Train: {np.mean(tr_accs):5.1f}% | Val: {np.mean(va_accs):5.1f}% | Gap: {np.mean(tr_accs) - np.mean(va_accs):+5.1f}%")
